# Implement Self-Attention from Scratch
### Problem Statement
Self-Attention is the mechanism where a sequence **attends to itself** — meaning Q, K, and V all come from the **same input**. This allows each token in a sequence to gather context from every other token.

Self-Attention is the core operation inside every Transformer encoder layer (e.g., BERT) and decoder layer (e.g., GPT), and understanding it is essential before moving to multi-head or cross-attention.

Your goal is to implement Self-Attention from scratch using PyTorch — projecting a single input `x` into Q, K, V, computing scaled dot-product attention, and producing the output.

---

### Requirements

1. **Linear Projections for Q, K, V from the same input**
   - Given a single input tensor `x`, project it into Q, K, and V using separate linear layers.

2. **Scaled Dot-Product Attention**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_model)`
   - Apply an optional `mask` before softmax (useful for causal/autoregressive masking).
   - Use the scores to weight `V`.

3. **Output Projection**
   - Apply a final linear projection to the attended output: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch's Reference**
   - Test your output against `torch.nn.MultiheadAttention` with `num_heads=1` using the same input for Q, K, V.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Q, K, V must all be derived from the **same input** `x`.
- ✅ Support optional masking (e.g., causal mask for autoregressive models).
- ✅ Must match `torch.nn.MultiheadAttention(num_heads=1)` output when Q=K=V=x.

---

<details>
  <summary>💡 Hint</summary>

  - The key difference from general attention: `q = k = v = x`. The same input is used for all three.
  - Softmax should be applied over the **last dimension** (attention scores across the sequence).
  - For causal masking, use `torch.tril()` to create a lower-triangular mask.
  - This is equivalent to calling `MultiheadAttention(x, x, x)` with `num_heads=1`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8

x = torch.rand(batch_size, seq_len, d_model)
print(x.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F




def self_attention(x, d_model, mask=None):
    """
    Implements self-attention (Q, K, V all derived from the same input).
    
    Args:
        x (Tensor): Input tensor of shape (batch_size, seq_len, d_model)
        d_model (int): Embedding dimension
        mask (Tensor, optional): Masking tensor for attention
        
    Returns:
        Tensor: Self-attention output of shape (batch_size, seq_len, d_model)
    """
    batch_size, seq_len, _ = x.shape
    
    # Linear projections for Q, K, V (all from the same input x)
    Q_w = nn.Linear(d_model, d_model, bias=False).to(device)
    K_w = nn.Linear(d_model, d_model, bias=False).to(device)
    V_w = nn.Linear(d_model, d_model, bias=False).to(device)
    W_out = nn.Linear(d_model, d_model, bias=False).to(device)

    Q = Q_w(x)  # (batch_size, seq_len, d_model)
    K = K_w(x)
    V = V_w(x)

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_model ** 0.5)  # (batch_size, seq_len, seq_len)

    # Mask check
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)  # (batch_size, seq_len, seq_len)

    output = torch.matmul(attn_weights, V)  # (batch_size, seq_len, d_model)

    return W_out(output)

In [ ]:
def self_attention(x, d_model, mask=None):
    batch_size, seq_len, _ = x.shape
    Q_w = nn.Linear(d_model,d_model,bias=False)
    K_w = nn.Linear(d_model,d_model,bias=False)
    V_w = nn.Linear(d_model,d_model,bias=False)
    W_out = nn.Linear(d_model,d_model,bias=False)

    Q = Q_w(x)
    K = K_w(x)
    V = V_w(x)

    # attention_score = QK/sqrt(d)
    attention_score = torch.matmul(Q,K.transpose(1,2))/(d_model ** 0.5)
    
    if mask is not None:
        attention_score = attention_score.masked_fill(mask==0, float('-inf'))


    attention_weights = F.softmax(attention_score,dim=-1)
    output = torch.matmul(attention_weights, V)
    output = W_out(output)
    return output

In [ ]:
# Testing on data & compare (without mask)
output_custom = self_attention(x, d_model)
print("Custom output:", output_custom.shape)
print(output_custom)

multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=1, bias=False, batch_first=True)
output, _ = multihead_attn(x, x, x)  # Self-attention: Q=K=V=x
print("\nPyTorch output:", output.shape)
print(output)

assert torch.allclose(output_custom, output, atol=1e-08, rtol=1e-05)  # Check if they are close enough.

In [ ]:
# Testing with causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len))  # Lower-triangular mask
print("Causal mask:")
print(causal_mask)

output_masked = self_attention(x, d_model, mask=causal_mask)
print("\nMasked output shape:", output_masked.shape)
print(output_masked)